In [0]:
-- ============================================================
-- FILE    : 01_bronze_schema.sql
-- LAYER   : Bronze
-- SCHEMA  : customer360
-- PURPOSE : CREATE TABLE statements for all 4 Bronze tables
--           Run this only if tables need to be recreated.
--           Normally tables are created by 01_bronze_initial_load.py
-- ============================================================

CREATE SCHEMA IF NOT EXISTS customer360
COMMENT 'Customer 360 — Bronze layer (raw ingested data)';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360.bronze_users
-- SOURCE: Synthetic — generated by 01_bronze_initial_load.py
-- GRAIN : One row per user signup event
-- NOTE  : Append-only. Dedup on user_id happens in Silver.
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360.bronze_users (
    user_id          STRING    NOT NULL  COMMENT 'Natural key — 8-char UUID prefix e.g. A1B2C3D4',
    name             STRING              COMMENT 'Full name — synthetic (Faker en_IN)',
    age              INT                 COMMENT 'Age at signup — range 18-65',
    gender           STRING              COMMENT 'Male / Female / Other',
    signup_date      STRING              COMMENT 'ISO date string YYYY-MM-DD',
    city             STRING              COMMENT 'Indian city — one of 10 cities',
    country          STRING              COMMENT 'Always India',
    email            STRING              COMMENT 'Synthetic email address',
    segment          STRING              COMMENT 'Premium / Standard / Basic / Trial',
    is_active        INT                 COMMENT '1 = active  0 = inactive',
    pipeline_run_id  STRING              COMMENT 'RUN_ID of pipeline that created this row — YYYYMMDD_HHMMSS or BOOTSTRAP_...'
)
USING DELTA
COMMENT 'Raw user signups — append-only, one row per pipeline run batch';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360.bronze_transactions
-- SOURCE: Synthetic — generated by 01_bronze_initial_load.py
-- GRAIN : One row per transaction event
-- NOTE  : Append-only. status can be success / failed / refunded.
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360.bronze_transactions (
    transaction_id         STRING    NOT NULL  COMMENT 'PK — TXN + 8-char UUID prefix e.g. TXNA1B2C3D4',
    user_id                STRING    NOT NULL  COMMENT 'FK → bronze_users.user_id',
    amount                 DOUBLE              COMMENT 'Gross transaction amount in INR — skewed distribution 50-50000',
    transaction_timestamp  STRING              COMMENT 'Event timestamp YYYY-MM-DD HH:MM:SS UTC',
    product_id             STRING              COMMENT 'Product reference — PRD + 4 digit code e.g. PRD1234',
    category               STRING              COMMENT 'Electronics / Fashion / Groceries / Travel / Entertainment / Health / Sports',
    payment_method         STRING              COMMENT 'Credit Card / Debit Card / UPI / Net Banking / Wallet',
    city                   STRING              COMMENT 'Transaction city — may differ from signup city',
    country                STRING              COMMENT 'Always India',
    status                 STRING              COMMENT 'success (85%) / failed (10%) / refunded (5%)',
    discount_pct           INT                 COMMENT 'Discount applied — 0 / 5 / 10 / 15 / 20',
    platform               STRING              COMMENT 'iOS / Android / Web',
    pipeline_run_id        STRING              COMMENT 'RUN_ID of pipeline that created this row'
)
USING DELTA
COMMENT 'Raw transaction events — append-only, one row per transaction';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360.bronze_app_usage
-- SOURCE: Synthetic — generated by 01_bronze_initial_load.py
-- GRAIN : One row per app session
-- NOTE  : is_bounce = 1 when session_duration_mins < 1
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360.bronze_app_usage (
    session_id             STRING    NOT NULL  COMMENT 'PK — SES + 8-char UUID prefix',
    user_id                STRING    NOT NULL  COMMENT 'FK → bronze_users.user_id',
    session_start          STRING              COMMENT 'Session start timestamp YYYY-MM-DD HH:MM:SS',
    session_end            STRING              COMMENT 'Session end timestamp YYYY-MM-DD HH:MM:SS',
    session_duration_mins  DOUBLE              COMMENT 'Duration in minutes — range 1.0-90.0',
    pages_visited          INT                 COMMENT 'Number of pages/screens visited — range 1-25',
    actions_taken          INT                 COMMENT 'Clicks, adds to cart, searches etc — range 0-15',
    device_type            STRING              COMMENT 'Mobile / Desktop / Tablet',
    platform               STRING              COMMENT 'iOS / Android / Web',
    is_bounce              INT                 COMMENT '1 = session under 60 seconds  0 = normal session',
    pipeline_run_id        STRING              COMMENT 'RUN_ID of pipeline that created this row'
)
USING DELTA
COMMENT 'Raw app session events — append-only, one row per session';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360.bronze_support_tickets
-- SOURCE: Synthetic — generated by 01_bronze_initial_load.py
-- GRAIN : One row per support ticket
-- NOTE  : resolved_at and resolution_hours are NULL for open tickets.
--         Heavy complainers (30 users) appear 3x more in ticket pool.
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360.bronze_support_tickets (
    ticket_id          STRING    NOT NULL  COMMENT 'PK — TKT + 8-char UUID prefix',
    user_id            STRING    NOT NULL  COMMENT 'FK → bronze_users.user_id',
    issue_type         STRING              COMMENT 'Payment Failure / Delivery Issue / Login Problem / Refund Request / Product Defect / Account Query',
    priority           STRING              COMMENT 'Low / Medium / High / Critical',
    created_at         STRING              COMMENT 'Ticket creation timestamp YYYY-MM-DD HH:MM:SS',
    resolved_at        STRING              COMMENT 'Resolution timestamp — NULL if still open',
    resolution_hours   DOUBLE              COMMENT 'Hours from creation to resolution — NULL if open',
    status             STRING              COMMENT 'open / closed',
    satisfaction_score INT                 COMMENT 'Customer rating 1-5 after resolution — NULL if open',
    pipeline_run_id    STRING              COMMENT 'RUN_ID of pipeline that created this row'
)
USING DELTA
COMMENT 'Raw support ticket events — append-only, one row per ticket';